In [ ]:
import os
import tempfile

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import torch

In [ ]:
Liver_sn_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq_raw.h5ad")
Liver_sn_raw

In [ ]:
Liver_sn_raw.obs['celltype'].value_counts()

In [ ]:
adata_stel = Liver_sn_raw[(Liver_sn_raw.obs['celltype'] == 'Stellete')].copy()
adata_stel

In [ ]:
adata_stel.raw = adata_stel
sc.pp.normalize_total(adata_stel, target_sum=1e4)
sc.pp.log1p(adata_stel)

In [ ]:
# 변수 유전자 설정 
sc.pp.highly_variable_genes(adata_stel, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(adata_stel, max_value=10)
# Run PCA
sc.tl.pca(adata_stel, svd_solver='arpack')
print(adata_stel.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_stel, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_stel.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
sc.pl.umap(
    adata_stel,
    color='Disease_status',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
# t-sne 
sc.tl.tsne(adata_stel, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_stel, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_stel)
sc.pl.umap(adata_stel, color=['sample'])

In [ ]:
sc.pl.umap(adata_stel, color=['journal'])

In [ ]:
print(adata_stel.obs["journal"].unique())
adata_stel.layers["counts"] = adata_stel.raw.X.copy()
print(np.isnan(adata_stel.X).sum(), "NaNs in X")
print(np.isinf(adata_stel.X).sum(), "Infs in X")

In [ ]:
scvi.model.SCVI.setup_anndata(adata_stel, batch_key="journal", layer="counts")

In [ ]:
scvi_ref = scvi.model.SCVI(
    adata_stel,
    use_layer_norm="both",
    use_batch_norm="none",
    encode_covariates=True,
    dropout_rate=0.2,
    n_layers=2,
)
scvi_ref.train()

In [ ]:
SCVI_LATENT_KEY = "X_scVI"

adata_stel.obsm[SCVI_LATENT_KEY] = scvi_ref.get_latent_representation()
sc.pp.neighbors(adata_stel, use_rep=SCVI_LATENT_KEY)
sc.tl.leiden(adata_stel)
sc.tl.umap(adata_stel)

In [ ]:
sc.pl.umap(
    adata_stel,
    color=["journal"],
    frameon=False,
    ncols=1,
)

In [ ]:
adata_stel.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_stellete_scvi.h5ad")

In [ ]:
sc.pl.violin(
    adata_stel,
    keys=['SLC7A5'],
    groupby='Disease_status',
    multi_panel=True,      # 유전자마다 subplot (세로로 쌓임)
    stripplot=False,       # 점 생략
    rotation=0
)